<a href="https://colab.research.google.com/github/karishmaram-tech/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karishmaram-tech/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one content item for one client on one date.

For the decision contract, the decision moment is the end of February 2026.

Features must use information available by the decision moment, using the February 2026 feature window. The outcome is measured during March 2026.

The March 2026 slice is also used for verification of the warehouse grain, date coverage, and data availability.

In [11]:
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Authenticate with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content"
DIM_CLIENTS = f"{REL}/dim_clients"

print("DuckDB connected.")
print("FlyRank warehouse paths configured.")

DuckDB connected.
FlyRank warehouse paths configured.


In [16]:
# Section 1 — verify the unit of analysis and March 2026 window

con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""")

┌───────────┬─────────┬───────────────┬────────────┬────────────┐
│ row_count │ clients │ content_items │ first_date │ last_date  │
│   int64   │  int64  │     int64     │    date    │    date    │
├───────────┼─────────┼───────────────┼────────────┼────────────┤
│   9841378 │      55 │        331437 │ 2026-03-01 │ 2026-03-31 │
└───────────┴─────────┴───────────────┴────────────┴────────────┘

### Features

The contract uses five features from the February 2026 feature window:

- `gsc_impressions` — search impressions available by the decision moment.
- `gsc_clicks` — search clicks available by the decision moment.
- `gsc_avg_position` — average search position available by the decision moment.
- `ga4_pageviews` — page views available by the decision moment when GA4 data is available.
- `ga4_engaged_sessions` — engaged sessions available by the decision moment when GA4 data is available.

Each feature is knowable at the decision moment because it is measured in the February feature window, before the March outcome window.

### Label

`target_declining` — whether the content item shows a decline in the March 2026 outcome window according to the contract's defined outcome rule.

The label is measured after the February decision moment and is therefore not available when the decision is made.

### Context

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`
- `gsc_data_available`
- `ga4_data_available`

These fields describe the observation or indicate whether source data is available.

### Excluded

- March outcome measurements used to construct the label.
- Any field calculated from the outcome window.
- Any label-derived field used as a model feature.
- Identifying client names, URLs, or private search queries.

### Deliberate leakage check

A deliberately label-derived feature was tested using the March outcome (`march_ctr`). The feature closely reproduces the March `target_declining` label, producing near-perfect agreement.

This demonstrates that outcome-window measurements cannot be used as features at the February decision moment. The leaked feature is therefore excluded from the final contract.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify that the five contracted features exist in the fact table

required_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

fact_columns = set(fact_cols["column_name"])

for feature in required_features:
    print(f"{feature}: {'FOUND' if feature in fact_columns else 'MISSING'}")

gsc_impressions: FOUND
gsc_clicks: FOUND
gsc_avg_position: FOUND
ga4_pageviews: FOUND
ga4_engaged_sessions: FOUND


In [25]:
# Build March 2026 outcome measurements at content-client level

march_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        CASE
            WHEN SUM(gsc_impressions) >= 100
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS march_ctr

    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
    GROUP BY
        client_hash_id,
        content_hash_id
""")

march_outcome.limit(5)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬───────────────────────┐
│     client_hash_id      │     content_hash_id      │ march_impressions │ march_clicks │       march_ctr       │
│         varchar         │         varchar          │      int128       │    int128    │        double         │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼───────────────────────┤
│ client_62f4a7e64f5e0096 │ content_39d7361b4945d504 │                77 │            0 │                  NULL │
│ client_62f4a7e64f5e0096 │ content_cec711b02f3bbde6 │               602 │            4 │  0.006644518272425249 │
│ client_62f4a7e64f5e0096 │ content_275b6f7f733016d4 │               810 │            1 │ 0.0012345679012345679 │
│ client_62f4a7e64f5e0096 │ content_ceaec531566ffcfc │                82 │            0 │                  NULL │
│ client_62f4a7e64f5e0096 │ content_755d951187fcd70a │              1858 │            6 

In [26]:
con.sql("""
    SELECT
        COUNT(*) AS content_items,
        COUNT(*) FILTER (WHERE march_ctr IS NOT NULL) AS eligible_items,
        MIN(march_ctr) AS min_ctr,
        MAX(march_ctr) AS max_ctr,
        AVG(march_ctr) AS average_ctr
    FROM march_outcome
""")

┌───────────────┬────────────────┬─────────┬─────────────────────┬───────────────────────┐
│ content_items │ eligible_items │ min_ctr │       max_ctr       │      average_ctr      │
│     int64     │     int64      │ double  │       double        │        double         │
├───────────────┼────────────────┼─────────┼─────────────────────┼───────────────────────┤
│        331437 │         101441 │     0.0 │ 0.15584415584415584 │ 0.0026157004304859194 │
└───────────────┴────────────────┴─────────┴─────────────────────┴───────────────────────┘

In [27]:
# Calculate the March CTR threshold for the outcome label

ctr_threshold = con.sql("""
    SELECT
        MEDIAN(march_ctr) AS median_march_ctr
    FROM march_outcome
    WHERE march_ctr IS NOT NULL
""").fetchone()[0]

print(f"March CTR threshold: {ctr_threshold:.6f}")
print(f"March CTR threshold (%): {ctr_threshold * 100:.3f}%")

March CTR threshold: 0.001238
March CTR threshold (%): 0.124%


In [28]:
# Create the March outcome label

march_labeled = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        march_impressions,
        march_clicks,
        march_ctr,

        CASE
            WHEN march_ctr IS NOT NULL
                 AND march_ctr < {ctr_threshold}
            THEN 1
            WHEN march_ctr IS NOT NULL
                 AND march_ctr >= {ctr_threshold}
            THEN 0
            ELSE NULL
        END AS target_declining

    FROM march_outcome
""")

march_labeled.limit(5)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬───────────────────────┬──────────────────┐
│     client_hash_id      │     content_hash_id      │ march_impressions │ march_clicks │       march_ctr       │ target_declining │
│         varchar         │         varchar          │      int128       │    int128    │        double         │      int32       │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼───────────────────────┼──────────────────┤
│ client_62f4a7e64f5e0096 │ content_d0dff76c889de68f │               181 │            0 │                   0.0 │                1 │
│ client_62f4a7e64f5e0096 │ content_67741cce996cfafa │                46 │            1 │                  NULL │             NULL │
│ client_62f4a7e64f5e0096 │ content_2e6360ad20fd7107 │               899 │            1 │ 0.0011123470522803114 │                1 │
│ client_62f4a7e64f5e0096 │ content_ac8663da7484669a │               

In [29]:
con.sql("""
    SELECT
        target_declining,
        COUNT(*) AS rows,
        ROUND(
            COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (),
            2
        ) AS percentage
    FROM march_labeled
    WHERE target_declining IS NOT NULL
    GROUP BY target_declining
    ORDER BY target_declining DESC
""")

┌──────────────────┬───────┬────────────┐
│ target_declining │ rows  │ percentage │
│      int32       │ int64 │   double   │
├──────────────────┼───────┼────────────┤
│                1 │ 50712 │      49.99 │
│                0 │ 50729 │      50.01 │
└──────────────────┴───────┴────────────┘

In [30]:
# Deliberate leakage experiment
# march_ctr is measured during the outcome window,
# so it would NOT be available at the February decision moment.

leakage_test = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        march_ctr,
        target_declining,

        CASE
            WHEN march_ctr < 0.0026157 THEN 1
            ELSE 0
        END AS leaked_feature

    FROM march_labeled
    WHERE target_declining IS NOT NULL
""")

leakage_test.limit(5)

┌─────────────────────────┬──────────────────────────┬───────────────────────┬──────────────────┬────────────────┐
│     client_hash_id      │     content_hash_id      │       march_ctr       │ target_declining │ leaked_feature │
│         varchar         │         varchar          │        double         │      int32       │     int32      │
├─────────────────────────┼──────────────────────────┼───────────────────────┼──────────────────┼────────────────┤
│ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │ 0.0010731258623332823 │                1 │              1 │
│ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │                   0.0 │                1 │              1 │
│ client_73cda7b4e4f265ea │ content_36c36abc7650d7af │ 0.0010657193605683837 │                1 │              1 │
│ client_73cda7b4e4f265ea │ content_a7da352b73b02668 │ 0.0026294498381877023 │                0 │              0 │
│ client_73cda7b4e4f265ea │ content_1855a661b4d36130 │  0.002331002331002331 │  

In [32]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN leaked_feature = target_declining THEN 1
                ELSE 0
            END
        ) AS matching_rows,

        ROUND(
            AVG(
                CASE
                    WHEN leaked_feature = target_declining THEN 1.0
                    ELSE 0.0
                END
            ) * 100,
            2
        ) AS agreement_percent

    FROM leakage_test
""")

┌────────────┬───────────────┬───────────────────┐
│ total_rows │ matching_rows │ agreement_percent │
│   int64    │    int128     │      double       │
├────────────┼───────────────┼───────────────────┤
│     101441 │         84927 │             83.72 │
└────────────┴───────────────┴───────────────────┘

The March 2026 performance slice contains 9,841,378 observations. The grain check shows that the total row count equals the number of unique client × content × date combinations, supporting the content-day observation definition.

The March slice runs from 2026-03-01 through 2026-03-31.

Data availability is uneven across rows. GSC data is available for 3,611,061 rows and GA4 data is available for 413,966 rows. Therefore, features depending on these sources should be interpreted with their availability flags and not assumed to exist for every observation.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 — verify the grain

con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
            AS unique_content_client_date_rows
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""")

┌────────────┬─────────────────────────────────┐
│ total_rows │ unique_content_client_date_rows │
│   int64    │              int64              │
├────────────┼─────────────────────────────────┤
│    9841378 │                         9841378 │
└────────────┴─────────────────────────────────┘

In [19]:
# Query 2 — verify March 2026 row count and date window

con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
    WHERE month = '2026-03'
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [20]:
# Query 3 — verify GSC and GA4 availability

con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS clients_with_gsc,
        COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS clients_with_ga4
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""")

┌────────────┬────────────────────┬────────────────────┬──────────────────┬──────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ clients_with_gsc │ clients_with_ga4 │
│   int64    │       int64        │       int64        │      int64       │      int64       │
├────────────┼────────────────────┼────────────────────┼──────────────────┼──────────────────┤
│    9841378 │            3611061 │             413966 │          9841378 │          6822637 │
└────────────┴────────────────────┴────────────────────┴──────────────────┴──────────────────┘

## 4. Data limits
This data supports observed and directional analysis of search and content performance, but it cannot establish that a content change caused a later performance change.

The history is not equally balanced across all sources. In the March verification slice, GSC and GA4 availability differ across observations, so source-dependent features should not be assumed to exist for every row.

The outcome is also defined over a separate March window from the February feature window. Therefore, the contract supports decision-support signals based on historical measurements, not causal conclusions about why performance changed.

The deliberate leakage test also shows that using outcome-window measurements as features can produce misleadingly strong results. Such fields must be excluded from the feature set.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summarize the source-availability limitation from the March verification slice

total_rows = 9841378
gsc_rows = 3611061
ga4_rows = 413966

print(f"GSC available: {gsc_rows:,} / {total_rows:,} rows "
      f"({gsc_rows / total_rows * 100:.2f}%)")

print(f"GA4 available: {ga4_rows:,} / {total_rows:,} rows "
      f"({ga4_rows / total_rows * 100:.2f}%)")

GSC available: 3,611,061 / 9,841,378 rows (36.69%)
GA4 available: 413,966 / 9,841,378 rows (4.21%)


In [21]:
# Inspect the content dimension columns

con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{DIM_CONTENT}.parquet')
""")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [22]:
# Show all dim_content columns clearly

con.sql(f"""
    SELECT column_name, column_type
    FROM (
        DESCRIBE SELECT *
        FROM read_parquet('{DIM_CONTENT}.parquet')
    )
    ORDER BY column_name
""")

┌────────────────────────────┬─────────────┐
│        column_name         │ column_type │
│          varchar           │   varchar   │
├────────────────────────────┼─────────────┤
│ backlinks                  │ BIGINT      │
│ category_count             │ BIGINT      │
│ char_count                 │ BIGINT      │
│ client_hash_id             │ VARCHAR     │
│ competition                │ DOUBLE      │
│ competition_level          │ VARCHAR     │
│ content_created_date       │ DATE        │
│ content_hash_id            │ VARCHAR     │
│ content_type               │ VARCHAR     │
│ content_updated_date       │ DATE        │
│          ·                 │  ·          │
│          ·                 │  ·          │
│          ·                 │  ·          │
│ keyword_token_count        │ BIGINT      │
│ last_optimized_date        │ DATE        │
│ main_intent                │ VARCHAR     │
│ model_used                 │ VARCHAR     │
│ optimization_eligible_date │ DATE        │
│ provider

In [23]:
# Check the fact table and content dimension for possible outcome/label fields

fact_cols = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""").df()

dim_cols = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{DIM_CONTENT}.parquet')
""").df()

print("FACT TABLE COLUMNS:")
print(fact_cols["column_name"].tolist())

print("\nDIM_CONTENT COLUMNS:")
print(dim_cols["column_name"].tolist())

FACT TABLE COLUMNS:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

DIM_CONTENT COLUMNS:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimizatio

### Self-check

- [x] Unit of analysis and time windows are defined.
- [x] Five or fewer features are specified.
- [x] Each feature is available before the decision moment.
- [x] The March outcome is kept separate from the February feature window.
- [x] Data availability was verified using the warehouse.
- [x] A deliberately label-derived feature was tested.
- [x] The leaked feature was excluded from the final contract.
- [x] Limitations are stated without causal claims.
- [x] Only anonymized identifiers and aggregated measurements are used.
- [x] No client names, URLs, private queries, or HF credentials are included.